In [2]:
import os

for dirname, _, filenames in os.walk("/kaggle/input"):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/competitions/house-prices-advanced-regression-techniques/sample_submission.csv
/kaggle/input/competitions/house-prices-advanced-regression-techniques/data_description.txt
/kaggle/input/competitions/house-prices-advanced-regression-techniques/train.csv
/kaggle/input/competitions/house-prices-advanced-regression-techniques/test.csv


In [3]:
import pandas as pd

train = pd.read_csv("/kaggle/input/competitions/house-prices-advanced-regression-techniques/train.csv")
test = pd.read_csv("/kaggle/input/competitions/house-prices-advanced-regression-techniques/test.csv")

print("Train shape:", train.shape)
print("Test shape:", test.shape)

Train shape: (1460, 81)
Test shape: (1459, 80)


In [4]:
train.head()

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


In [3]:
import pandas as pd

train = pd.read_csv("/kaggle/input/competitions/house-prices-advanced-regression-techniques/train.csv")
test = pd.read_csv("/kaggle/input/competitions/house-prices-advanced-regression-techniques/test.csv")

print("Train:", train.shape)
print("Test:", test.shape)

print("\nSalePrice:")
print(train["SalePrice"].describe())

Train: (1460, 81)
Test: (1459, 80)

SalePrice:
count      1460.000000
mean     180921.195890
std       79442.502883
min       34900.000000
25%      129975.000000
50%      163000.000000
75%      214000.000000
max      755000.000000
Name: SalePrice, dtype: float64


In [6]:
X = train.drop("SalePrice", axis=1)
y = train["SalePrice"]

print("Features:", X.shape)
print("Target:", y.shape)

Features: (1460, 80)
Target: (1460,)


In [7]:
missing = X.isnull().sum()

print(missing[missing > 0].sort_values(ascending=False).head(10))

PoolQC          1453
MiscFeature     1406
Alley           1369
Fence           1179
MasVnrType       872
FireplaceQu      690
LotFrontage      259
GarageType        81
GarageYrBlt       81
GarageFinish      81
dtype: int64


In [8]:
# Combine train and test so both have the same columns
all_data = pd.concat([X, test.drop("Id", axis=1)], axis=0)

# Fill numeric missing values with median
numeric_cols = all_data.select_dtypes(include=["int64", "float64"]).columns
all_data[numeric_cols] = all_data[numeric_cols].fillna(
    all_data[numeric_cols].median()
)

# Fill text/categorical missing values with most common value
categorical_cols = all_data.select_dtypes(include=["object"]).columns
all_data[categorical_cols] = all_data[categorical_cols].fillna(
    all_data[categorical_cols].mode().iloc[0]
)

# Convert categorical columns into numbers
all_data = pd.get_dummies(all_data)

print("Final data shape:", all_data.shape)
print("Missing values left:", all_data.isnull().sum().sum())

Final data shape: (2919, 288)
Missing values left: 0


In [2]:
import pandas as pd
from sklearn.ensemble import RandomForestRegressor

# Load data
train = pd.read_csv("/kaggle/input/competitions/house-prices-advanced-regression-techniques/train.csv")
test = pd.read_csv("/kaggle/input/competitions/house-prices-advanced-regression-techniques/test.csv")

# Separate target and ID
y = train["SalePrice"]
X = train.drop(["SalePrice", "Id"], axis=1)
test_data = test.drop("Id", axis=1)

# Combine train and test
all_data = pd.concat([X, test_data], axis=0)

# Fill numerical missing values
numeric_cols = all_data.select_dtypes(include=["int64", "float64"]).columns
all_data[numeric_cols] = all_data[numeric_cols].fillna(
    all_data[numeric_cols].median()
)

# Fill categorical missing values
categorical_cols = all_data.select_dtypes(include=["object"]).columns
all_data[categorical_cols] = all_data[categorical_cols].fillna(
    all_data[categorical_cols].mode().iloc[0]
)

# Convert categorical data into numbers
all_data = pd.get_dummies(all_data)

# Split back into train and test
X_processed = all_data.iloc[:len(train)]
test_processed = all_data.iloc[len(train):]

# Create and train model
model = RandomForestRegressor(
    n_estimators=200,
    random_state=42
)

model.fit(X_processed, y)

print("Model trained successfully!")
print("Training data:", X_processed.shape)
print("Test data:", test_processed.shape)

Model trained successfully!
Training data: (1460, 287)
Test data: (1459, 287)


In [3]:
predictions = model.predict(test_processed)

print("Predictions created!")
print(predictions[:10])

Predictions created!
[129696.54  156078.87  182682.65  179977.7   198201.115 183959.65
 166465.365 175898.485 183798.235 119449.205]


In [4]:
submission = pd.DataFrame({
    "Id": test["Id"],
    "SalePrice": predictions
})

submission.to_csv("/kaggle/working/submission.csv", index=False)

print("Submission file created!")
print(submission.head())

Submission file created!
     Id   SalePrice
0  1461  129696.540
1  1462  156078.870
2  1463  182682.650
3  1464  179977.700
4  1465  198201.115
